# AER - Access Entitlement Review Reporter v4.2

**Updates (v4.2):**
- **Bug Fix**: Cache hit stats no longer inflate (per-row calculation instead of total).
- **Global Report**: New `📊 Global Report` button exports summary across all apps.
- **User Info**: Per-App reports now include User Name & User Email columns.
- **File Paths**: All reports saved under `output/{date}/report/`, logs under `report/logs/`.
- **Safe Write**: Auto-appends `_1`, `_2`... if target file is locked/open.

**Previous Features:**
- **Smart Tab Handling**: Prioritizes "User Listing" tab.
- **Full Data Caching**: Reports populated from cache are identical to live reads.
- **Merged Logic**: Parsing logic integrated into Engine.
- **Full Audit History & Advanced Reporting**.

In [ ]:
# === Cell 1: Setup & Authentication ===
import os
import sys
import io
import logging
from datetime import datetime
from dotenv import load_dotenv
from msal import PublicClientApplication
import requests

load_dotenv()

# === Logging ===
today_str = datetime.now().strftime('%Y-%m-%d')
hour_str = datetime.now().strftime('%H')
log_dir = f"output/{today_str}/report/logs"
os.makedirs(log_dir, exist_ok=True)

log_file = f"{log_dir}/aer_{today_str}_{hour_str}00.log"

logger = logging.getLogger("aer")
logger.handlers.clear()
logger.setLevel(logging.INFO)

def _get_console_stream():
    s = getattr(sys, "stdout", None)
    try:
        if s and hasattr(s, "reconfigure"):
            s.reconfigure(encoding="utf-8")
            if hasattr(sys.stderr, "reconfigure"):
                sys.stderr.reconfigure(encoding="utf-8")
            return s
    except Exception: pass
    return sys.stdout

formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
ch = logging.StreamHandler(_get_console_stream())
ch.setFormatter(formatter)
logger.addHandler(ch)

fh = logging.FileHandler(log_file, encoding="utf-8", mode='a')
fh.setFormatter(formatter)
logger.addHandler(fh)

# === Azure AD Config ===
TENANT_ID = os.getenv("AZURE_TENANT_ID")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID")
SHAREPOINT_HOST = os.getenv("SHAREPOINT_HOST", "davidshih.sharepoint.com")
SITE_NAME = os.getenv("SITE_NAME", "aer")
APP_NAME = "2025 Entitlement Review"
BASE_PATH = APP_NAME
SENDER_EMAIL = os.getenv("SENDER_EMAIL")

SCOPES = [
    "User.Read.All", "Mail.Send", "Mail.Read", 
    "Files.Read.All", "Sites.Read.All"
]

app = PublicClientApplication(CLIENT_ID, authority=f"https://login.microsoftonline.com/{TENANT_ID}")
print("🚀 Opening browser for authentication...")
interactive_result = app.acquire_token_interactive(scopes=SCOPES, prompt="select_account")

if "access_token" not in interactive_result:
    raise RuntimeError(f"Login Failed: {interactive_result.get('error_description')}")

headers = {"Authorization": f"Bearer {interactive_result['access_token']}"}
logger.info("✅ Login Successful")
logger.info(f"Log File: {log_file}")

In [ ]:
# === Cell 2: SharePoint API & App Selector (v3.6 with Counts & Filtering) ===
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests

# --- 1. SharePoint API Functions ---
def get_site_id(site_name: str) -> str:
    url = f"https://graph.microsoft.com/v1.0/sites/{SHAREPOINT_HOST}:/sites/{site_name}"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()["id"]

def list_folders_with_count(site_id: str, path: str) -> list[dict]:
    if not path or path.strip() == "":
        url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root/children"
    else:
        clean_path = path.strip("/")
        url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/children"
    
    resp = requests.get(url, headers=headers)
    results = []
    EXCLUDED_NAMES = ["forms", "_private", "audit logs", "audit", "user listings", "shared documents"]

    for item in resp.json().get("value", []):
        if item.get("folder"):
            name_lower = item["name"].lower()
            if any(ex in name_lower for ex in EXCLUDED_NAMES):
                continue

            count = item.get("folder", {}).get("childCount", 0)
            results.append({
                "name": item["name"], 
                "webUrl": item.get("webUrl", ""),
                "count": count
            })
    return results

def list_excel_files(site_id: str, folder_path: str) -> list[dict]:
    clean_path = folder_path.strip("/")
    url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/children"
    resp = requests.get(url, headers=headers)
    files = []
    for item in resp.json().get("value", []):
        if item["name"].endswith(".xlsx"):
            files.append({
                "id": item["id"], "name": item["name"],
                "lastModifiedDateTime": item.get("lastModifiedDateTime"),
                "createdDateTime": item.get("createdDateTime"),
                "webUrl": item.get("webUrl")
            })
    return sorted(files, key=lambda f: f.get("lastModifiedDateTime", ""), reverse=True)

def download_file(site_id: str, file_path: str) -> bytes:
    clean_path = file_path.strip("/")
    url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/root:/{clean_path}:/content"
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.content

def get_file_audit_info(site_id: str, file_id: str) -> dict:
    url = f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive/items/{file_id}/versions"
    resp = requests.get(url, headers=headers)
    default_res = {"log": "N/A", "creator": "Unknown", "modifier": "Unknown", "created_ts": None}
    if resp.status_code != 200: return default_res
    
    versions = resp.json().get("value", [])
    if not versions: return default_res

    logs = []
    for v in versions:
        mod_time = v.get("lastModifiedDateTime", "")[:19].replace("T", " ")
        actor = v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System"
        logs.append(f"{mod_time} - {actor}")
    
    last_v = versions[0]
    first_v = versions[-1]

    return {
        "log": "\n".join(logs),
        "creator": first_v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System",
        "modifier": last_v.get("lastModifiedBy", {}).get("user", {}).get("displayName") or "System",
        "created_ts": first_v.get("lastModifiedDateTime")
    }

# --- 2. Initialize Connection ---
try:
    site_id = get_site_id(SITE_NAME)
    logger.info(f"✅ SharePoint Connected (Site ID: {site_id[:10]}...)")
except Exception as e:
    logger.error(f"❌ Connection Failed: {e}")

# --- 3. Tree Selector UI (with Counts) ---
TARGET_APPS = [] 
USE_CACHE = True 

def create_app_selector():
    print(f"📂 Reading Root: {BASE_PATH} ...")
    try:
        categories = list_folders_with_count(site_id, BASE_PATH)
    except Exception as e:
        print(f"❌ Read Failed: {e}")
        return

    ui_container = widgets.VBox()
    app_checkboxes = []
    app_checkbox_map = {}
    
    chk_cache = widgets.Checkbox(value=True, description="⚡ Use Cache (Faster)", indent=False)
    ui_container.children += (widgets.HTML("<h3>📂 App Selector (v3.6)</h3>"), chk_cache, widgets.HTML("<hr>"))

    for cat in categories:
        if cat['name'] in ["Forms", "_private"] or "audit" in cat['name'].lower(): continue
            
        cat_label = widgets.HTML(f"<b>📁 {cat['name']}</b>", layout=widgets.Layout(width='150px'))
        btn_expand = widgets.Button(description="➕ Expand", button_style='info', layout=widgets.Layout(width='80px'))
        pbar = widgets.IntProgress(value=0, min=0, max=1, layout=widgets.Layout(width='160px', visibility='hidden'))
        app_list_box = widgets.VBox(layout=widgets.Layout(margin='0 0 0 30px', display='none'))
        
        def on_expand(b, cat_name=cat['name'], container=app_list_box, btn=btn_expand, pbar=pbar):
            if btn.description == "➕ Expand":
                btn.description = "⏳"
                pbar.layout.visibility = 'visible'
                pbar.bar_style = 'info'
                pbar.value = 0
                pbar.max = 1
                sub_path = f"{BASE_PATH}/{cat_name}"
                try:
                    apps = list_folders_with_count(site_id, sub_path)
                    apps = [a for a in apps if a.get('name') not in ["Forms", "_private"]]
                    checks = []
                    if not apps:
                        checks.append(widgets.Label("(Empty)"))
                        pbar.value = 1
                    else:
                        pbar.max = len(apps)
                        for idx, app in enumerate(apps, 1):
                            count_display = f" <span style='color:#888; font-size:11px'>({app['count']} users)</span>"
                            app_key = f"{cat_name}|{app['name']}"
                            cb = app_checkbox_map.get(app_key)
                            if cb is None:
                                cb = widgets.Checkbox(value=False, description=app['name'], indent=False, layout=widgets.Layout(width='400px'))
                                app_checkbox_map[app_key] = cb
                                app_checkboxes.append(cb)
                            cb.description = app['name']
                            cb.app_data = (cat_name, app['name'], f"{sub_path}/{app['name']}")
                            lbl_count = widgets.HTML(count_display)
                            checks.append(widgets.HBox([cb, lbl_count]))
                            pbar.value = idx
                    container.children = tuple(checks)
                    container.layout.display = 'block'
                    btn.description = "➖ Collapse"
                except Exception as e:
                    container.children = (widgets.Label(f"Error: {e}"),)
                    btn.description = "❌"
                finally:
                    pbar.layout.visibility = 'hidden'
                    pbar.bar_style = ''
            else:
                if container.layout.display == 'none':
                    container.layout.display = 'block'; btn.description = "➖ Collapse"
                else:
                    container.layout.display = 'none'; btn.description = "➕ Expand"

        btn_expand.on_click(on_expand)
        ui_container.children += (widgets.HBox([btn_expand, cat_label, pbar]), app_list_box)

    btn_confirm = widgets.Button(description="✅ Confirm Selection", button_style='success', layout=widgets.Layout(margin='20px 0'))
    output_area = widgets.Output()

    def on_confirm(b):
        global TARGET_APPS, USE_CACHE
        TARGET_APPS = [cb.app_data for cb in app_checkboxes if cb.value]
        USE_CACHE = chk_cache.value
        
        with output_area:
            clear_output()
            if not TARGET_APPS: print("⚠️ No apps selected!")
            else:
                print(f"🎯 Selected {len(TARGET_APPS)} Apps | Cache: {USE_CACHE}")
                print("⏳ Proceed to Cell 4 (Enrichment) or Cell 5 (Scan).")

    btn_confirm.on_click(on_confirm)
    display(ui_container, btn_confirm, output_area)

create_app_selector()

In [ ]:
# === Cell 5: AER Engine v4.2 (Cache Fix, User Info, Global Report, Safe Write) ===

import pandas as pd

import re, time, json, os, requests

import ipywidgets as widgets

from datetime import datetime

from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

from io import BytesIO

from IPython.display import display, HTML, clear_output

import importlib.util
from IPython import get_ipython

def _widgets_env_ready():
    try:
        ip = get_ipython()
        if ip is None or ip.__class__.__name__ != 'ZMQInteractiveShell':
            return False
        has_nb = importlib.util.find_spec('widgetsnbextension') is not None
        has_lab = importlib.util.find_spec('jupyterlab_widgets') is not None
        return has_nb or has_lab
    except Exception:
        return False

USE_WIDGETS = _widgets_env_ready()


# ==========================================

# PART 1: CONFIG & LOADER

# ==========================================

CACHE_FILE = "aer_cache.json"

NOTES_FILE = "aer_manual_notes.json"

CACHE_VERSION = 4.2

cache_updated = False

EXCLUDED_FOLDERS = ["forms", "_private", "user listings", "audit logs", "audit"]



def load_json(file_path):

    if os.path.exists(file_path):

        try:

            with open(file_path, 'r', encoding='utf-8') as f: return json.load(f)

        except: return {}

    return {}



def save_json(file_path, data):

    with open(file_path, 'w', encoding='utf-8') as f: json.dump(data, f, indent=2, ensure_ascii=False)



def safe_excel_path(base_path):

    """If base_path is locked/open, append _1, _2, ... until writable."""

    if not os.path.exists(base_path):

        return base_path

    try:

        with open(base_path, 'a'):

            pass

        return base_path

    except OSError:

        pass

    stem, ext = os.path.splitext(base_path)

    for i in range(1, 100):

        candidate = f"{stem}_{i}{ext}"

        if not os.path.exists(candidate):

            return candidate

    return f"{stem}_{int(time.time())}{ext}"



def format_export_excel(file_path, audit_col_name="Audit Log"):

    wb = load_workbook(file_path)

    ws = wb.active

    header_to_col = {}

    for col_idx in range(1, ws.max_column + 1):

        hdr = ws.cell(row=1, column=col_idx).value

        hdr_txt = str(hdr).strip() if hdr is not None else ""

        if hdr_txt:

            header_to_col[hdr_txt] = col_idx

        max_len = len(hdr_txt)

        for row_idx in range(2, ws.max_row + 1):

            cell_val = ws.cell(row=row_idx, column=col_idx).value

            if cell_val is None:

                continue

            lines = str(cell_val).splitlines() or [str(cell_val)]

            max_len = max(max_len, max(len(line) for line in lines))

        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(10, max_len + 2), 80)

    audit_col = header_to_col.get(audit_col_name)

    if audit_col:

        for row_idx in range(2, ws.max_row + 1):

            cell = ws.cell(row=row_idx, column=audit_col)

            txt = "" if cell.value is None else str(cell.value)

            line_count = max(1, txt.count("\n") + 1)

            cell.alignment = Alignment(wrap_text=True, vertical="top")

            current_height = ws.row_dimensions[row_idx].height or 15

            ws.row_dimensions[row_idx].height = max(current_height, min(15 * line_count, 300))

    wb.save(file_path)



local_cache = load_json(CACHE_FILE)

manual_data_store = load_json(NOTES_FILE)



# ==========================================

# PART 2: ROBUST EXCEL PARSER

# ==========================================

def find_col_index(headers, keywords):

    """Case-insensitive search for column index."""

    for idx, h in enumerate(headers):

        if not h: continue

        h_str = str(h).strip().lower()

        if all(k in h_str for k in keywords):

            return idx

    return None



def is_name_like_header(header_text: str) -> bool:

    txt = (header_text or "").strip().lower()

    if not txt:

        return False

    if "reviewer" in txt or "manager" in txt:

        return False

    return ("name" in txt) or ("display" in txt and "user" in txt) or ("full" in txt and "name" in txt)



def is_email_like_header(header_text: str) -> bool:

    txt = (header_text or "").strip().lower()

    if not txt:

        return False

    return ("email" in txt) or ("mail" == txt) or txt.endswith(" mail")



def resolve_column_map(header, app_col_map=None):

    if app_col_map:

        return app_col_map, "app-locked"

    idx_rev = find_col_index(header, ["reviewer"])

    idx_res = find_col_index(header, ["response"])

    idx_det = find_col_index(header, ["details", "change"])

    idx_user = None

    idx_email = None

    if idx_rev is not None:

        rev_hdr = str(header[idx_rev]).strip().lower() if header[idx_rev] else ""

        if "response" in rev_hdr:

            idx_rev = None

            for i, h in enumerate(header):

                h_str = str(h).strip().lower() if h else ""

                if ("reviewer" in h_str) and ("response" not in h_str):

                    idx_rev = i

                    break

    for i in range(min(2, len(header))):

        h = str(header[i]).strip().lower() if header[i] else ""

        if idx_user is None and is_name_like_header(h):

            idx_user = i

        if idx_email is None and is_email_like_header(h):

            idx_email = i

    if idx_user is None: idx_user = find_col_index(header, ["user", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["display", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["full", "name"])

    if idx_user is None: idx_user = find_col_index(header, ["name"])

    if idx_email is None: idx_email = find_col_index(header, ["email"])

    if idx_email is None: idx_email = find_col_index(header, ["mail"])

    if idx_rev is None: idx_rev = find_col_index(header, ["manager"])

    if idx_rev is None or idx_res is None:

        return None, "invalid"

    return {

        "idx_rev": idx_rev, "idx_res": idx_res, "idx_det": idx_det,

        "idx_user": idx_user, "idx_email": idx_email

    }, "detected"



def read_excel_rows(excel_bytes: bytes, reviewer_name: str, file_name: str, folder_url: str, app_col_map=None):

    wb = load_workbook(BytesIO(excel_bytes), read_only=True)



    # 1. Smart Tab Selection

    sheet_name = wb.sheetnames[0]

    for sn in wb.sheetnames:

        if "user listing" in sn.lower(): sheet_name = sn; break

    ws = wb[sheet_name]



    # 2. Read Headers

    rows_iter = ws.iter_rows(values_only=True)

    try:

        header = next(rows_iter)

    except StopIteration:

        return [], None, "empty-sheet"



    # 3. Column Mapping

    col_map, map_source = resolve_column_map(header, app_col_map=app_col_map)

    if not col_map:

        return [], None, map_source

    idx_rev = col_map["idx_rev"]

    idx_res = col_map["idx_res"]

    idx_det = col_map["idx_det"]

    idx_user = col_map["idx_user"]

    idx_email = col_map["idx_email"]



    results = []



    # 4. Iterate Rows

    for i, row in enumerate(rows_iter, start=2):

        r_rev = row[idx_rev] if idx_rev < len(row) else None

        r_res = row[idx_res] if idx_res < len(row) else None

        r_det = row[idx_det] if idx_det is not None and idx_det < len(row) else None

        r_user = row[idx_user] if idx_user is not None and idx_user < len(row) else None

        r_email = row[idx_email] if idx_email is not None and idx_email < len(row) else None



        if str(r_rev).strip().lower() != reviewer_name.lower(): continue



        val_res = str(r_res).strip() if r_res else ""

        val_det = str(r_det).strip() if r_det else ""



        results.append({

            "reviewer": reviewer_name,

            "user_name": str(r_user).strip() if r_user else "",

            "user_email": str(r_email).strip() if r_email else "",

            "response": val_res,

            "details": val_det,

            "is_missing": (val_res == ""),

            "row_number": i,

            "file_name": file_name,

            "folder_url": folder_url

        })

    return results, col_map, map_source



def get_row_stats(txt):

    txt = str(txt).lower().strip()

    kw_appr = ['approv', 'retain', 'keep', 'confirm', 'yes', 'ok', 'active']

    kw_deny = ['denied', 'deny', 'remove', 'delete', 'revok', 'reject', 'no']

    kw_chg  = ['change', 'modif', 'updat', 'correct', 'edit', 'adjust']

    return {

        "is_appr": 1 if any(k in txt for k in kw_appr) else 0,

        "is_deny": 1 if any(k in txt for k in kw_deny) else 0,

        "is_chg":  1 if any(k in txt for k in kw_chg) else 0

    }



# ==========================================

# PART 3: SCANNING ENGINE

# ==========================================

if 'TARGET_APPS' not in globals() or not TARGET_APPS:

    print("⚠️ Please select Apps in Cell 2 first!")

    TARGET_APPS = []



all_responses = []

errors = []

app_column_map_store = {}



print(f"🚀 Starting Scan Engine v{CACHE_VERSION} (Fuzzy Column Match)...")



# UI: Per-app scan progress (details stay in log file)
scan_progress_box = widgets.VBox()
scan_progress_store = {}

if TARGET_APPS:
    scan_rows = []
    for category, app_name, app_path in TARGET_APPS:
        ui_key = f"{category}|{app_name}|{app_path}"
        w_label = widgets.HTML(
            f"<b>{category} &gt; {app_name}</b>",
            layout=widgets.Layout(width="360px"),
        )
        w_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=1,
            bar_style="",
            layout=widgets.Layout(width="260px"),
        )
        w_status = widgets.HTML("Queued", layout=widgets.Layout(width="120px"))
        scan_progress_store[ui_key] = {"bar": w_bar, "status": w_status}
        scan_rows.append(
            widgets.HBox([w_label, w_bar, w_status], layout=widgets.Layout(align_items="center"))
        )
    scan_progress_box.children = tuple(scan_rows)
    display(widgets.VBox([widgets.HTML("<h4>📡 Scan Progress (Per App)</h4>"), scan_progress_box]))


for category, current_app_name, current_path in TARGET_APPS:

    ui_key = f"{category}|{current_app_name}|{current_path}"
    app_ui = scan_progress_store.get(ui_key)

    try:

        if app_ui:
            app_ui["bar"].bar_style = "info"
            app_ui["bar"].value = 0
            app_ui["bar"].max = 1
            app_ui["status"].value = "Loading..."

        raw_folders = list_folders_with_count(site_id, current_path) if 'list_folders_with_count' in globals() else []

        if not raw_folders: raw_folders = [{"name": "Error", "webUrl": "#"}]



        reviewers = [r for r in raw_folders if r['name'].lower() not in EXCLUDED_FOLDERS]

        total_revs = len(reviewers)

        if app_ui:
            app_ui["bar"].max = max(1, total_revs)
            app_ui["bar"].value = 0
            app_ui["status"].value = f"0/{total_revs}"

        logger.info(f"📂 App: {current_app_name} | Reviewers: {total_revs}")

        app_schema_key = f"{category}|{current_app_name}"

        app_col_map = app_column_map_store.get(app_schema_key)



        from concurrent.futures import ThreadPoolExecutor, as_completed

        def _scan_one(folder, idx, app_col_map):
            reviewer_name = folder["name"]
            folder_url = folder["webUrl"]
            folder_path = f"{current_path}/{reviewer_name}"
            cache_key = f"{category}|{current_app_name}|{reviewer_name}"

            try:
                files = list_excel_files(site_id, folder_path)
                match_candidates = [f for f in files if reviewer_name.lower() in f["name"].lower()]
                target_file = match_candidates[0] if match_candidates else (files[0] if files else None)

                if not target_file:
                    logger.info(f"  ⚠️ Skip: [{idx}/{total_revs}] {reviewer_name} (no xlsx found)")
                    return {"skip": True, "idx": idx, "reviewer": reviewer_name}

                remote_mod = target_file.get("lastModifiedDateTime")
                logger.info(f"  🔎 File: [{idx}/{total_revs}] {reviewer_name} | Target:{target_file['name']} | Modified:{remote_mod}")

                cached = local_cache.get(cache_key)
                is_hit = False
                force_live_recheck = False
                cache_pending = False

                if USE_CACHE and cached and cached.get('v') == CACHE_VERSION and cached.get('last_mod') == remote_mod and 'rows' in cached and len(cached['rows']) > 0:
                    cache_pending = any(r.get('is_missing') for r in cached.get('rows', []))
                    if cache_pending:
                        force_live_recheck = True
                        logger.info(f"  ♻️ Cache Recheck: [{idx}/{total_revs}] {reviewer_name} has pending rows; reading live and overwriting cache.")
                    else:
                        is_hit = True
                        logger.info(f"  🧊 Cache Hit (Completed): [{idx}/{total_revs}] {reviewer_name} | rows:{len(cached['rows'])}")
                else:
                    if not USE_CACHE:
                        logger.info(f"  ℹ️ Cache Bypass: [{idx}/{total_revs}] {reviewer_name} (USE_CACHE=False)")
                    elif not cached:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (no cache key)")
                    elif cached.get('v') != CACHE_VERSION:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (version {cached.get('v')} != {CACHE_VERSION})")
                    elif cached.get('last_mod') != remote_mod:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (modified changed)")
                    else:
                        logger.info(f"  ℹ️ Cache Miss: [{idx}/{total_revs}] {reviewer_name} (no cached rows)")

                if is_hit:
                    audit_snap = cached.get('audit', {})
                    c_appr = cached['stats']['Appr']
                    c_deny = cached['stats']['Deny']
                    c_chg = cached['stats']['Chg']
                    c_miss = sum(1 for r in cached['rows'] if r['is_missing'])

                    logger.info(f"  ✅ Read (Cache): [{idx}/{total_revs}] {reviewer_name} (Missing:{c_miss})(A:{c_appr}, D:{c_deny}, C:{c_chg})")

                    rows_out = []
                    for r in cached['rows']:
                        r_copy = r.copy()
                        st = get_row_stats(r['response'])
                        r_copy.update({
                            "Category": category, "App_Name": current_app_name,
                            "Last_Modified": remote_mod, "File_Created_Date": audit_snap.get('created_ts'),
                            "Audit_Log": audit_snap.get('log'), "File_Creator": audit_snap.get('creator'), "File_Modifier": audit_snap.get('modifier'),
                            "stats_appr": st['is_appr'], "stats_deny": st['is_deny'], "stats_chg": st['is_chg'],
                            "source_is_cache": True
                        })
                        rows_out.append(r_copy)

                    return {
                        "rows": rows_out,
                        "cache_entry": None,
                        "cache_key": cache_key,
                        "detected_col_map": None,
                        "map_source": None,
                        "idx": idx,
                        "total": total_revs,
                        "reviewer": reviewer_name,
                        "folder_url": folder_url,
                        "stats": {"Appr": c_appr, "Deny": c_deny, "Chg": c_chg, "Miss": c_miss},
                        "force_live_recheck": False,
                        "read_mode": "cache"
                    }

                content = download_file(site_id, f"{folder_path}/{target_file['name']}")
                audit_info = get_file_audit_info(site_id, target_file["id"])

                rows, detected_col_map, map_source = read_excel_rows(
                    content, reviewer_name, target_file['name'], folder_url, app_col_map=app_col_map
                )

                s_appr, s_deny, s_chg, miss_cnt = 0, 0, 0, 0
                clean_rows_cache = []
                final_created = audit_info.get('created_ts') or target_file.get("createdDateTime")

                for r in rows:
                    st = get_row_stats(r['response'])
                    s_appr += st['is_appr']; s_deny += st['is_deny']; s_chg += st['is_chg']
                    if r['is_missing']: miss_cnt += 1

                    clean_rows_cache.append({
                        "reviewer": r['reviewer'], "user_name": r.get('user_name', ''), "user_email": r.get('user_email', ''),
                        "response": r['response'], "details": r['details'],
                        "is_missing": r['is_missing'], "row_number": r['row_number'],
                        "file_name": r['file_name'], "folder_url": r['folder_url']
                    })

                    r.update({
                        "Category": category, "App_Name": current_app_name,
                        "Last_Modified": remote_mod, "File_Created_Date": final_created,
                        "Audit_Log": audit_info['log'], "File_Creator": audit_info['creator'], "File_Modifier": audit_info['modifier'],
                        "stats_appr": st['is_appr'], "stats_deny": st['is_deny'], "stats_chg": st['is_chg'],
                        "source_is_cache": False
                    })

                logger.info(
                    f"  ✅ Read (Live): [{idx}/{total_revs}] {reviewer_name} "
                    f"(Rows:{len(rows)} Missing:{miss_cnt})(A:{s_appr}, D:{s_deny}, C:{s_chg})"
                )

                cache_entry = None
                if rows or force_live_recheck:
                    cache_entry = {
                        "v": CACHE_VERSION, "last_mod": remote_mod,
                        "stats": {"Appr": s_appr, "Deny": s_deny, "Chg": s_chg},
                        "audit": audit_info, "rows": clean_rows_cache
                    }

                return {
                    "rows": rows,
                    "cache_entry": cache_entry,
                    "cache_key": cache_key,
                    "detected_col_map": detected_col_map,
                    "map_source": map_source,
                    "idx": idx,
                    "total": total_revs,
                    "reviewer": reviewer_name,
                    "folder_url": folder_url,
                    "stats": {"Appr": s_appr, "Deny": s_deny, "Chg": s_chg, "Miss": miss_cnt,
                              "CacheRows": len(clean_rows_cache), "ForceRecheck": force_live_recheck},
                    "force_live_recheck": force_live_recheck,
                    "read_mode": "live"
                }
            except Exception as e:
                return {"error": str(e), "reviewer": reviewer_name, "folder_url": folder_url, "idx": idx, "total": total_revs}

        done_count = 0
        max_workers = 4
        futures = []
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for idx, folder in enumerate(reviewers, 1):
                futures.append(executor.submit(_scan_one, folder, idx, app_col_map))

            for fut in as_completed(futures):
                result = fut.result()
                if result.get("skip"):
                    done_count += 1
                elif result.get("error"):
                    logger.error(f"  ❌ Error {result.get('reviewer', '')}: {result['error']}")
                    errors.append({"Category": category, "App_Name": current_app_name, "reviewer": result.get("reviewer", ""), "error": result["error"], "folder_url": result.get("folder_url", "#")})
                    done_count += 1
                else:
                    if result.get("read_mode") == "live":
                        detected_col_map = result.get("detected_col_map")
                        map_source = result.get("map_source")
                        if detected_col_map and not app_col_map:
                            app_col_map = detected_col_map
                            app_column_map_store[app_schema_key] = detected_col_map
                            logger.info(
                                f"  🧭 App Column Map Locked: NameIdx={detected_col_map.get('idx_user')} "
                                f"EmailIdx={detected_col_map.get('idx_email')} Source={map_source}"
                            )
                        elif detected_col_map and map_source == "app-locked":
                            logger.info(
                                f"  🧭 App Column Map Reused: NameIdx={detected_col_map.get('idx_user')} "
                                f"EmailIdx={detected_col_map.get('idx_email')}"
                            )
                        elif not detected_col_map:
                            logger.warning(f"  ⚠️ Column Mapping Failed: [{result['idx']}/{result['total']}] {result['reviewer']}")

                    if result.get("rows"):
                        all_responses.extend(result["rows"])

                    cache_entry = result.get("cache_entry")
                    if cache_entry is not None:
                        local_cache[result["cache_key"]] = cache_entry
                        cache_updated = True
                        stats = result.get("stats", {})
                        logger.info(
                            f"  💾 Cache Write: [{result['idx']}/{result['total']}] {result['reviewer']} "
                            f"(Rows:{stats.get('CacheRows', 0)} Missing:{stats.get('Miss', 0)} PendingRecheck:{stats.get('ForceRecheck', False)})"
                        )

                    done_count += 1

                if app_ui:
                    app_ui["bar"].value = min(done_count, app_ui["bar"].max)
                    app_ui["status"].value = f"{done_count}/{total_revs}"
        if app_ui:
            app_ui["bar"].bar_style = "success"
            if total_revs <= 0:
                app_ui["bar"].value = 1
                app_ui["bar"].max = 1
                app_ui["status"].value = "0/0 Done"
            else:
                app_ui["bar"].value = total_revs
                app_ui["status"].value = f"{total_revs}/{total_revs} Done"

    except Exception as e:

        logger.error(f"❌ App Error: {e}")

        if app_ui:

            app_ui["bar"].bar_style = "danger"

            app_ui["status"].value = "Error"



if cache_updated:

    save_json(CACHE_FILE, local_cache)

    logger.info("💾 Cache Updated (v4.2)")



# ==========================================

# PART 4: DASHBOARD

# ==========================================

df = pd.DataFrame(all_responses)

widget_store = {}

unified_data = {}

today_str = datetime.now().strftime("%Y-%m-%d")

report_dir = f"output/{today_str}/report"

os.makedirs(report_dir, exist_ok=True)



if not df.empty:

    stats = df.groupby(["Category", "App_Name", "reviewer"]).agg(

        total_rows=("reviewer", "size"),

        missing=("is_missing", "sum"),

        approved=("stats_appr", "sum"), denied=("stats_deny", "sum"), changed=("stats_chg", "sum"),

        f_creator=("File_Creator", "first"), f_modifier=("File_Modifier", "first"), audit=("Audit_Log", "first"),

        is_cached=("source_is_cache", "all")

    ).reset_index()



    for _, row in stats.iterrows():

        key = f"{row['Category']} > {row['App_Name']}"

        if key not in unified_data:

            saved_app = manual_data_store.get(key, {})

            unified_data[key] = {

                "Category": row['Category'], "App_Name": row['App_Name'],

                "status_manual": saved_app.get("app_status", "Calculated"), "note_manual": saved_app.get("app_note", ""),

                "reviewers": {}, "stats": {"total_users": 0, "completed_users": 0}

            }



        node = unified_data[key]

        node['stats']['total_users'] += 1

        is_done = (row['missing'] == 0)

        if is_done: node['stats']['completed_users'] += 1

        status_calc = f"❌ Pending: {row['missing']}"

        if is_done: status_calc = "✅ Cached - Completed" if row['is_cached'] else "✅ Completed"



        d_style = "color:red;font-weight:bold" if row['denied'] > 0 else "color:#555"

        node['reviewers'][row['reviewer']] = {

            "status_calc": status_calc,

            "detail_html": f"Appr:{int(row['approved'])} | <span style='{d_style}'>Deny:{int(row['denied'])}</span> | Chg:{int(row['changed'])}",

            "folder_url": df[(df['App_Name'] == row['App_Name']) & (df['reviewer'] == row['reviewer'])].iloc[0].get('folder_url', '#')

        }



def _compute_overall_progress(stats_df, raw_df):

    folder_map = raw_df.groupby(["Category", "App_Name", "reviewer"]).agg(
        folder_url=("folder_url", "first"),
    ).reset_index()

    merged = stats_df.merge(folder_map, on=["Category", "App_Name", "reviewer"], how="left")

    total_app_reviewers = int(len(merged))
    pending_app_reviewers = int((merged["missing"] > 0).sum()) if total_app_reviewers else 0
    done_app_reviewers = total_app_reviewers - pending_app_reviewers
    progress_pct = round(done_app_reviewers / total_app_reviewers * 100, 1) if total_app_reviewers else 0.0
    distinct_reviewers_to_notify = int(merged.loc[merged["missing"] > 0, "reviewer"].nunique()) if total_app_reviewers else 0
    distinct_apps_remaining = int(merged.loc[merged["missing"] > 0, "App_Name"].nunique()) if total_app_reviewers else 0

    remaining = merged[merged["missing"] > 0].copy()
    if not remaining.empty:
        denom = remaining["total_rows"].replace(0, pd.NA)
        remaining["Completion %"] = (((remaining["total_rows"] - remaining["missing"]) / denom) * 100).round(1).fillna(0.0)
        remaining = remaining.rename(columns={
            "App_Name": "App Name",
            "reviewer": "Reviewer",
            "missing": "Missing",
            "total_rows": "Total Rows",
            "folder_url": "Folder URL",
        })
        remaining = remaining[["Category", "App Name", "Reviewer", "Missing", "Total Rows", "Completion %", "Folder URL"]]
        remaining = remaining.sort_values(["Missing", "Category", "App Name", "Reviewer"], ascending=[False, True, True, True])
    else:
        remaining = pd.DataFrame(columns=["Category", "App Name", "Reviewer", "Missing", "Total Rows", "Completion %", "Folder URL"])

    overview_df = pd.DataFrame([
        {"Metric": "Total app-reviewers", "Value": total_app_reviewers},
        {"Metric": "Pending app-reviewers", "Value": pending_app_reviewers},
        {"Metric": "Done app-reviewers", "Value": done_app_reviewers},
        {"Metric": "Overall progress (%)", "Value": progress_pct},
        {"Metric": "Distinct reviewers to notify", "Value": distinct_reviewers_to_notify},
        {"Metric": "Distinct app to be completed", "Value": distinct_apps_remaining},
    ])

    return {
        "total_app_reviewers": total_app_reviewers,
        "pending_app_reviewers": pending_app_reviewers,
        "done_app_reviewers": done_app_reviewers,
        "progress_pct": progress_pct,
        "distinct_reviewers_to_notify": distinct_reviewers_to_notify,
        "distinct_apps_remaining": distinct_apps_remaining,
        "overview_df": overview_df,
        "remaining_work_df": remaining,
    }


def _build_remaining_work_compact_html(remaining_df, sort_by="app"):

    if remaining_df is None or remaining_df.empty:
        return "<i>✅ No pending app-reviewers. All done.</i>"

    df = remaining_df.copy()
    if sort_by == "reviewer":
        sort_cols = ["Reviewer", "Category", "App Name"]
    else:
        sort_cols = ["App Name", "Reviewer", "Category"]
    df = df.sort_values(sort_cols, kind="stable")

    item_style = "margin:2px 0; font-size:12px; line-height:1.4"
    items = []
    for _, r in df.iterrows():
        label = f"{r.get('Category', '')} / {r.get('App Name', '')} - {r.get('Reviewer', '')}"
        url = r.get("Folder URL") or "#"
        items.append(
            f"<div style='{item_style}'>"
            f"<a href='{url}' target='_blank' rel='noopener noreferrer'>{label}</a>"
            "</div>"
        )
    return "<div style='width:96%; margin:0 auto'>" + "".join(items) + "</div>"


def _build_remaining_work_html(remaining_df):

    if remaining_df is None or remaining_df.empty:
        return "<i>✅ No pending app-reviewers. All done.</i>"

    cell_style = "padding:6px; overflow-wrap:anywhere; word-break:break-word; white-space:normal; vertical-align:middle"
    table = (
        "<div style='width:100%; overflow:auto'>"
        "<table border='1' style='margin:0 auto; border-collapse:collapse; width:96%; font-size:12px; "
        "border:1px solid #d8dde6; table-layout:fixed'>"
    )
    table += (
        "<tr style='background:#eef3f8; font-weight:bold'>"
        "<th style='padding:6px; width:18%; vertical-align:middle'>Quarter</th>"
        "<th style='padding:6px; width:24%; vertical-align:middle'>App Name</th>"
        "<th style='padding:6px; width:20%; vertical-align:middle'>Reviewer</th>"
        "<th style='padding:6px; width:10%; vertical-align:middle'>Missing</th>"
        "<th style='padding:6px; width:10%; vertical-align:middle'>Total</th>"
        "<th style='padding:6px; width:10%; vertical-align:middle'>Completion %</th>"
        "<th style='padding:6px; width:8%; vertical-align:middle'>Link</th>"
        "</tr>"
    )

    for _, r in remaining_df.iterrows():
        url = r.get("Folder URL") or "#"
        link_html = (
            f"<a href='{url}' target='_blank' rel='noopener noreferrer' "
            "style='background:#0078d4;color:#fff;padding:6px 10px;border-radius:6px;"
            "text-decoration:none;display:inline-block;white-space:nowrap'>Open Folder</a>"
        )
        table += (
            "<tr>"
            f"<td style='{cell_style}'>{r.get('Category', '')}</td>"
            f"<td style='{cell_style}'>{r.get('App Name', '')}</td>"
            f"<td style='{cell_style}'>{r.get('Reviewer', '')}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{int(r.get('Missing', 0))}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{int(r.get('Total Rows', 0))}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{r.get('Completion %', 0)}</td>"
            f"<td style='padding:6px; text-align:center; vertical-align:middle'>{link_html}</td>"
            "</tr>"
        )

    table += "</table></div>"
    return table


def _build_progress_overview_html(total_app_reviewers, pending_app_reviewers, done_app_reviewers, progress_pct, distinct_reviewers_to_notify, distinct_apps_remaining, remaining_df):
    bar_pct = 0.0 if total_app_reviewers == 0 else round(done_app_reviewers / total_app_reviewers * 100, 1)
    metrics_html = (
        f"<b>Total app-reviewers:</b> {total_app_reviewers} &nbsp; "
        f"<b>Pending:</b> {pending_app_reviewers} &nbsp; "
        f"<b>Done:</b> {done_app_reviewers} &nbsp; "
        f"<b>Progress:</b> {progress_pct}% &nbsp; "
        f"<b>Distinct apps remaining:</b> {distinct_apps_remaining} &nbsp; "
        f"<b>Distinct reviewers remaining:</b> {distinct_reviewers_to_notify}"
    )
    bar_html = (
        "<div style='width:420px; height:12px; border:1px solid #cfd7e6; border-radius:6px; overflow:hidden; display:inline-block; vertical-align:middle'>"
        f"<div style='height:12px; width:{bar_pct}%; background:#4caf50'></div>"
        "</div>"
        f"<span style='margin-left:8px; font-size:12px; color:#555'>{done_app_reviewers}/{total_app_reviewers} completed</span>"
    )
    if remaining_df is None or remaining_df.empty:
        remaining_body = _build_remaining_work_compact_html(remaining_df, "app")
    else:
        remaining_by_app = _build_remaining_work_compact_html(remaining_df, "app")
        remaining_by_reviewer = _build_remaining_work_compact_html(remaining_df, "reviewer")
        remaining_body = (
            "<div style='margin-top:6px'><b>Sort by App</b><br>"
            f"{remaining_by_app}</div>"
            "<div style='margin-top:10px'><b>Sort by Reviewer</b><br>"
            f"{remaining_by_reviewer}</div>"
        )
    remaining_html = (
        "<details open>"
        "<summary style='font-weight:bold; cursor:pointer'>🧾 Remaining Work (Pending App-Reviewers)</summary>"
        f"{remaining_body}"
        "</details>"
    )
    return (
        "<h4>Progress Overview</h4>"
        f"{metrics_html}<br><br>{bar_html}"
        f"{remaining_html}"
    )

def _build_summary_table_html(stats_df, raw_df):
    if stats_df is None or stats_df.empty:
        return "<i>No app-reviewer data.</i>"
    folder_map = raw_df.groupby(["Category", "App_Name", "reviewer"]).agg(folder_url=("folder_url", "first")).reset_index()
    merged = stats_df.merge(folder_map, on=["Category", "App_Name", "reviewer"], how="left")
    merged["Status"] = merged["missing"].apply(lambda x: "Completed" if x == 0 else "Pending")
    merged["Folder"] = merged["folder_url"].fillna("#").apply(
        lambda u: f"<a href='{u}' target='_blank' rel='noopener noreferrer'>Open Folder</a>"
    )
    show_cols = ["Category", "App_Name", "reviewer", "Status", "missing", "total_rows", "approved", "denied", "changed", "Folder"]
    renamed = merged[show_cols].rename(columns={
        "App_Name": "App Name",
        "reviewer": "Reviewer",
        "missing": "Missing",
        "total_rows": "Total Rows",
        "approved": "Approved",
        "denied": "Denied",
        "changed": "Changed",
    })
    return renamed.to_html(index=False, escape=False)

def build_dashboard():

    container = widgets.VBox()

    btn_export = widgets.Button(description="💾 Save Reports", button_style='success', icon='file-excel')

    btn_global = widgets.Button(description="📊 Global Report", button_style='primary')

    lbl_out = widgets.Label()



    prog = _compute_overall_progress(stats, df)
    total_app_reviewers = prog["total_app_reviewers"]
    pending_app_reviewers = prog["pending_app_reviewers"]
    done_app_reviewers = prog["done_app_reviewers"]
    progress_pct = prog["progress_pct"]
    distinct_reviewers_to_notify = prog["distinct_reviewers_to_notify"]
    distinct_apps_remaining = prog["distinct_apps_remaining"]
    remaining_df = prog["remaining_work_df"]

    if not USE_WIDGETS:
        overview_html = _build_progress_overview_html(
            total_app_reviewers,
            pending_app_reviewers,
            done_app_reviewers,
            progress_pct,
            distinct_reviewers_to_notify,
            distinct_apps_remaining,
            remaining_df,
        )
        summary_html = _build_summary_table_html(stats, df)
        display(HTML(overview_html + "<h4>All App-Reviewers</h4>" + summary_html))
        return

    w_metrics = widgets.HTML(
        f"<b>Total app-reviewers:</b> {total_app_reviewers} &nbsp; "
        f"<b>Pending:</b> {pending_app_reviewers} &nbsp; "
        f"<b>Done:</b> {done_app_reviewers} &nbsp; "
        f"<b>Progress:</b> {progress_pct}% &nbsp; "
        f"<b>Distinct apps remaining:</b> {distinct_apps_remaining} &nbsp; "
        f"<b>Distinct reviewers remaining:</b> {distinct_reviewers_to_notify}"
    )
    w_bar = widgets.IntProgress(
        value=done_app_reviewers,
        min=0,
        max=max(1, total_app_reviewers),
        bar_style="success" if pending_app_reviewers == 0 else "info",
        layout=widgets.Layout(width="420px"),
    )
    w_bar_lbl = widgets.HTML(
        f"{done_app_reviewers}/{total_app_reviewers} completed",
        layout=widgets.Layout(width="180px"),
    )

    w_remaining_html = widgets.HTML(_build_remaining_work_compact_html(remaining_df, "app"))
    w_sort = widgets.ToggleButtons(
        options=[("App", "app"), ("Reviewer", "reviewer")],
        value="app",
        description="Sort by",
        layout=widgets.Layout(width="320px"),
    )
    def _on_sort_change(change):
        if change.get("name") == "value":
            w_remaining_html.value = _build_remaining_work_compact_html(remaining_df, change.get("new"))
    w_sort.observe(_on_sort_change, names="value")
    w_remaining_box = widgets.VBox([w_sort, w_remaining_html], layout=widgets.Layout(margin="4px 0 0 0"))
    w_remaining_acc = widgets.Accordion(children=[w_remaining_box])
    w_remaining_acc.set_title(0, "🧾 Remaining Work (Pending App-Reviewers)")
    w_remaining_acc.selected_index = 0

    overview_box = widgets.VBox([
        widgets.HTML("<h4>📈 Progress Overview</h4>"),
        w_metrics,
        widgets.HBox([w_bar, w_bar_lbl], layout=widgets.Layout(align_items="center")),
        w_remaining_acc,
    ], layout=widgets.Layout(margin="8px 0 12px 0"))



    app_widgets = []

    for app_key in sorted(unified_data.keys()):

        app_data = unified_data[app_key]

        pct = int((app_data['stats']['completed_users'] / app_data['stats']['total_users'] * 100)) if app_data['stats']['total_users'] > 0 else 0

        w_lbl = widgets.HTML(f"<b>📂 {app_key}</b> &nbsp; <span style='background:#eee; padding:2px 5px; border-radius:4px'>{pct}% Done</span>", layout=widgets.Layout(width='400px'))

        w_stat = widgets.Dropdown(options=["Calculated", "Force Completed", "Action Required"], value=app_data['status_manual'], layout=widgets.Layout(width='150px'))

        widget_store[app_key] = {"data": app_data, "w_stat": w_stat}



        rev_list = widgets.VBox([

            widgets.HBox([

                widgets.HTML(f"<a href='{rd['folder_url']}' target='_blank'>{rn}</a>", layout=widgets.Layout(width='250px')),

                widgets.HTML(rd['status_calc'], layout=widgets.Layout(width='200px')),

                widgets.HTML(rd['detail_html'])

            ]) for rn, rd in app_data['reviewers'].items()

        ], layout=widgets.Layout(margin='5px 0 5px 20px', display='none'))



        btn_tog = widgets.Button(description="➕ Show Users", layout=widgets.Layout(width='100px'))

        def create_tog(w):

            def on_tog(b):

                if w.layout.display == 'none': w.layout.display = 'block'; b.description = "➖ Hide"

                else: w.layout.display = 'none'; b.description = "➕ Show Users"

            return on_tog

        btn_tog.on_click(create_tog(rev_list))

        app_widgets.append(widgets.VBox([widgets.HBox([btn_tog, w_lbl, w_stat]), rev_list]))



    def export(b):

        b.disabled=True; b.description="Saving..."

        saved_files = []

        for app_key, widget_data in widget_store.items():

            app_name = widget_data['data']['App_Name']

            category = widget_data['data']['Category']

            app_rows = df[(df['Category'] == category) & (df['App_Name'] == app_name)].to_dict('records')



            final_data = []

            for row in app_rows:

                overwrite_st = widget_data['w_stat'].value

                if overwrite_st == "Calculated": overwrite_st = ""



                final_data.append({

                    "User Name": row.get('user_name', ''),

                    "User Email": row.get('user_email', ''),

                    "Reviewer": row['reviewer'],

                    "File Name": row.get('file_name'),

                    "Reviewer Response": row.get('response'),

                    "Details of Access Change": row.get('details'),

                    "Overwrite Status": overwrite_st,

                    "Row Num": row.get('row_number'),

                    "Audit Log": row.get('Audit_Log')

                })



            if final_data:

                safe_name = re.sub(r'[\\/*?:"<>|]', "", app_name)

                f_path = safe_excel_path(f"{report_dir}/{safe_name}_{today_str}.xlsx")

                pd.DataFrame(final_data).to_excel(f_path, index=False)
                format_export_excel(f_path)

                saved_files.append(safe_name)



        lbl_out.value = f"Saved {len(saved_files)} files."

        b.disabled=False; b.description="💾 Save Reports"



    def export_global(b):

        b.disabled = True; b.description = "Saving..."

        prog = _compute_overall_progress(stats, df)

        rows = []

        for _, r in stats.sort_values("App_Name").iterrows():

            progress = "Completed" if r['missing'] == 0 else f"{int(r['missing'])}/{int(r['total_rows'])} missing"

            rows.append({

                "Category": r['Category'],

                "App Name": r['App_Name'],

                "Reviewer": r['reviewer'],

                "Progress": progress,

                "Total Approved": int(r['approved']),

                "Total Denied": int(r['denied']),

                "Total Changed": int(r['changed'])

            })

        f_path = safe_excel_path(f"{report_dir}/Summary_Report_{today_str}.xlsx")

        summary_df = pd.DataFrame(rows)
        overview_df = prog["overview_df"]
        remaining_df = prog["remaining_work_df"]

        with pd.ExcelWriter(f_path, engine="openpyxl") as writer:
            summary_df.to_excel(writer, index=False, sheet_name="Sheet1")
            overview_df.to_excel(writer, index=False, sheet_name="Progress", startrow=0)
            start = len(overview_df) + 2
            remaining_df.to_excel(writer, index=False, sheet_name="Progress", startrow=start)

        format_export_excel(f_path)

        lbl_out.value = f"Global report saved."

        b.disabled = False; b.description = "📊 Global Report"



    btn_export.on_click(export)

    btn_global.on_click(export_global)

    container.children = tuple([widgets.HBox([btn_export, btn_global, lbl_out]), overview_box] + app_widgets)

    display(container)



if unified_data:

    build_dashboard()

else:

    print("⚠️ No data found.")


In [ ]:
# === CELL 7: Stage 7 — Email Notification Center ===
# Send review reminders to reviewers. Uses AD cache for email lookup (not per-reviewer Graph API).
# Configurable template footer. Batch checkpoint to avoid double-send.

# ============================================
# EMAIL HELPERS
# ============================================

EMAIL_CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "email_sent.json")

def _normalize_reviewer_name(value):
    if pd.isna(value):
        return ""
    cleaned = re.sub(r"[^\w\s]", "", str(value).strip().lower())
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

def _build_identity_index_cell7(df):
    email_set = set()
    name_to_emails = {}
    if df is None:
        return email_set, name_to_emails
    for _, row in df.iterrows():
        email = str(row.get("email", "")).strip().lower()
        if not email or email == "nan":
            continue
        email_set.add(email)
        name = _normalize_reviewer_name(row.get("displayName", ""))
        if name:
            name_to_emails.setdefault(name, set()).add(email)
    return email_set, name_to_emails

def _resolve_identity_cell7(value, ad_email_set, ad_name_map):
    """Resolve reviewer to email with punctuation-insensitive name matching."""
    if pd.isna(value):
        return False, "", "Value is blank/NaN"
    raw = str(value).strip()
    if not raw:
        return False, "", "Value is blank"
    if any(ch in raw for ch in BRACKET_CHARS):
        return False, "", "Value contains bracket characters"
    lower_val = raw.lower()
    if "@" in lower_val:
        if not is_email_valid(lower_val):
            return False, "", "Email format is invalid"
        if lower_val not in ad_email_set:
            corrected, was_corrected = correct_email_domain(lower_val)
            if was_corrected and corrected in ad_email_set:
                return True, corrected, ""
            return False, "", "Email not found in AD"
        return True, lower_val, ""
    normalized = _normalize_reviewer_name(raw)
    matches = sorted(list(ad_name_map.get(normalized, set())))
    if len(matches) == 0:
        return False, "", "Name not found in AD"
    if len(matches) > 1:
        return False, "", f"Name maps to {len(matches)} AD accounts"
    return True, matches[0], ""

def _email_lookup_from_ad(reviewer_name, ad_email_set=None, ad_name_map=None):
    """Resolve reviewer name to email using AD cache (no Graph API call)."""
    if ad_email_set is None or ad_name_map is None:
        ad_df, _, _ = load_ad_cache()
        if ad_df is not None:
            ad_email_set, ad_name_map = _build_identity_index_cell7(ad_df)
        else:
            return ""
    ok, email, _ = _resolve_identity_cell7(reviewer_name, ad_email_set, ad_name_map)
    return email if ok else ""

def _fmt_date_long(iso_str):
    if not iso_str or pd.isna(iso_str):
        return "Unknown Date"
    try:
        dt_str = str(iso_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        return dt.strftime("%B %d, %Y")
    except Exception:
        return str(iso_str)

def _calc_due_date(iso_str, days=14):
    if not iso_str or pd.isna(iso_str):
        return "ASAP"
    try:
        dt_str = str(iso_str).replace("Z", "").split(".")[0]
        dt = datetime.fromisoformat(dt_str)
        return (dt + timedelta(days=days)).strftime("%B %d, %Y")
    except Exception:
        return "ASAP"

def _infer_section(file_date_raw):
    if not file_date_raw or pd.isna(file_date_raw):
        return "followup"
    try:
        dt_str = str(file_date_raw).replace("Z", "").split(".")[0]
        sent_dt = datetime.fromisoformat(dt_str)
        return "new" if (sent_dt + timedelta(days=14)) >= datetime.now() else "followup"
    except Exception:
        return "followup"

def _build_section_table(rows):
    table = "<table border='1' style='border-collapse:collapse; width:100%; font-size:12px; border:1px solid #ddd'>"
    table += "<tr style='background:#f3f3f3'><th>App Name</th><th>Sent Date</th><th>Due Date</th><th>Missing</th><th>Link</th></tr>"
    for row in rows:
        table += (
            "<tr>"
            f"<td style='padding:5px'>{row['App_Name']}</td>"
            f"<td style='padding:5px'>{row['sent_date']}</td>"
            f"<td style='padding:5px'>{row['due_date']}</td>"
            f"<td style='padding:5px; text-align:center'>{row['missing']}</td>"
            f"<td style='padding:5px'><a href='{row['folder_url']}'>Open</a></td>"
            "</tr>"
        )
    table += "</table>"
    return table

def _build_email_html(reviewer_name, new_rows, followup_rows):
    parts = [
        f"Hi {reviewer_name},<br><br>",
        "The Information Security team is sending you an entitlement review update.<br><br>",
    ]
    if new_rows:
        parts.append("<b>1) New Applications to Review</b><br>")
        parts.append(_build_section_table(new_rows))
        parts.append("<br><br>")
    if followup_rows:
        parts.append("<b>2) Follow-up on Remaining Reviews</b><br>")
        parts.append(_build_section_table(followup_rows))
        parts.append("<br><br>")
    if not new_rows and not followup_rows:
        parts.append("No applications are selected for this update.<br><br>")
    parts.append("Please complete these reviews as soon as possible. Your prompt assistance is appreciated.<br><br>")
    # Configurable footer
    footer_html = AER_EMAIL_TEMPLATE_FOOTER.replace("\n", "<br>")
    parts.append(footer_html)
    return "".join(parts)


# ============================================
# DATA PREP
# ============================================

s7_status = widgets.HTML(value="<i>Preparing email data...</i>")
s7_output = widgets.Output()

s7_container = widgets.VBox()
s7_row_store = {}

s7_subject = widgets.Text(
    value="2026 User entitlement Reviews",
    description="<b>Subject:</b>",
    layout=widgets.Layout(width="98%"),
)
s7_cc = widgets.Textarea(
    value="", description="<b>Global CC:</b>",
    placeholder="manager@company.com, team@company.com",
    layout=widgets.Layout(width="98%", height="50px"),
)
s7_reply_to = widgets.Text(
    value="", description="<b>Reply-To:</b>",
    placeholder="security-team@company.com",
    layout=widgets.Layout(width="98%"),
)
s7_btn_refresh = widgets.Button(description="🔄 Refresh", button_style="info",
                                 layout=widgets.Layout(width="120px"))
s7_btn_send_all = widgets.Button(description="🔥 Send All (Batch)", button_style="danger",
                                  layout=widgets.Layout(width="200px"))


def _parse_email_list(raw):
    if not raw:
        return []
    return [p.strip() for p in re.split(r"[,\n;]+", str(raw)) if p and p.strip()]


def _render_email_rows(b=None):
    global s7_row_store
    s7_row_store = {}
    s7_container.children = (widgets.Label("Loading..."),)

    df = globals().get("r6_df")
    if df is None or df.empty:
        s7_container.children = (widgets.HTML("<h4>⚠️ No scan data. Run Stage 6 first.</h4>"),)
        return

    pending = df[df["is_missing"] == True].copy()
    if pending.empty:
        s7_container.children = (widgets.HTML("<h3>✅ No pending emails!</h3>"),)
        return

    targets = pending.groupby(["Category", "App_Name", "reviewer", "folder_url"]).agg(
        missing_count=("is_missing", "count"),
        file_date_raw=("File_Created_Date", "min"),
    ).reset_index()

    # Resolve emails using AD cache (not Graph API per reviewer!)
    ad_df, _, _ = load_ad_cache()
    ad_es, ad_nm = _build_identity_index_cell7(ad_df) if ad_df is not None else (set(), {})
    email_cache = {}
    sent_log = load_json_safe(EMAIL_CHECKPOINT_FILE)

    raw_list = []
    for _, r in targets.iterrows():
        rev = r["reviewer"]
        if rev not in email_cache:
            email_cache[rev] = _email_lookup_from_ad(rev, ad_es, ad_nm)
        raw_list.append({
            "App_Name": r["App_Name"], "reviewer": rev,
            "missing": int(r["missing_count"]),
            "folder_url": r["folder_url"],
            "sent_date": _fmt_date_long(r["file_date_raw"]),
            "due_date": _calc_due_date(r["file_date_raw"]),
            "file_date_raw": r["file_date_raw"],
            "email": email_cache[rev],
        })

    df_raw = pd.DataFrame(raw_list).sort_values(["reviewer", "App_Name"]).reset_index(drop=True)
    ui_items = []

    for idx, ((reviewer, email), group) in enumerate(df_raw.groupby(["reviewer", "email"], dropna=False)):
        key = f"{reviewer}_{idx}"

        # Check if already sent
        already_sent = sent_log.get(key, {}).get("sent", False)

        email_color = "#0078d4" if email else "red"
        w_chk = widgets.Checkbox(value=not already_sent, layout=widgets.Layout(width="30px"))
        w_info = widgets.HTML(
            f"<b>👤 {reviewer}</b><br><span style='color:{email_color}'>{email or '(No Email)'}</span>"
            + (" ✅ Sent" if already_sent else ""),
            layout=widgets.Layout(width="260px"),
        )
        w_email = widgets.Text(value=email or "", placeholder="Email", layout=widgets.Layout(width="98%"))
        w_btn = widgets.Button(description="🚀 Send", button_style="warning" if not already_sent else "success",
                               layout=widgets.Layout(width="90px"))

        app_data = []
        for _, app_row in group.iterrows():
            section = _infer_section(app_row["file_date_raw"])
            app_data.append({
                "App_Name": app_row["App_Name"], "missing": int(app_row["missing"]),
                "folder_url": app_row["folder_url"], "sent_date": app_row["sent_date"],
                "due_date": app_row["due_date"], "section": section,
            })

        s7_row_store[key] = {
            "reviewer": reviewer, "w_chk": w_chk, "w_email": w_email, "w_btn": w_btn,
            "apps": app_data,
        }

        # Summary text
        new_count = sum(1 for a in app_data if a["section"] == "new")
        followup_count = len(app_data) - new_count
        summary = widgets.HTML(
            f"<span style='color:#0b6'>New: {new_count}</span> | "
            f"<span style='color:#b36b00'>Follow-up: {followup_count}</span>"
        )

        def make_sender(k):
            def _send(_):
                state = s7_row_store[k]
                to_email = state["w_email"].value.strip()
                if not to_email:
                    state["w_btn"].description = "No Email"
                    return
                state["w_btn"].disabled = True
                state["w_btn"].description = "..."
                try:
                    new_rows = [a for a in state["apps"] if a["section"] == "new"]
                    fu_rows = [a for a in state["apps"] if a["section"] == "followup"]
                    body = _build_email_html(state["reviewer"], new_rows, fu_rows)
                    subj = s7_subject.value

                    headers = token_mgr.get_headers("graph")
                    url = f"https://graph.microsoft.com/v1.0/users/{SENDER_EMAIL}/sendMail"
                    to_list = [{"emailAddress": {"address": to_email}}]
                    cc = [{"emailAddress": {"address": e}} for e in _parse_email_list(s7_cc.value)]
                    reply_to = [{"emailAddress": {"address": e}} for e in _parse_email_list(s7_reply_to.value)]
                    msg = {"subject": subj, "body": {"contentType": "HTML", "content": body}, "toRecipients": to_list}
                    if cc:
                        msg["ccRecipients"] = cc
                    if reply_to:
                        msg["replyTo"] = reply_to

                    resp = graph_post(url, headers, json_payload={"message": msg})
                    if resp.status_code == 202:
                        state["w_btn"].button_style = "success"
                        state["w_btn"].description = "Sent"
                        state["w_chk"].value = False
                        # Checkpoint
                        sl = load_json_safe(EMAIL_CHECKPOINT_FILE)
                        sl[k] = {"sent": True, "ts": datetime.now().isoformat(), "email": to_email}
                        atomic_json_save(EMAIL_CHECKPOINT_FILE, sl)
                    else:
                        state["w_btn"].button_style = "danger"
                        state["w_btn"].description = "Fail"
                        with s7_output:
                            print(f"Send failed for {state['reviewer']}: {resp.text[:200]}")
                except Exception as e:
                    state["w_btn"].button_style = "danger"
                    state["w_btn"].description = "Err"
                    with s7_output:
                        print(f"Error: {e}")
                finally:
                    if state["w_btn"].description != "Sent":
                        state["w_btn"].disabled = False
            return _send

        w_btn.on_click(make_sender(key))

        ui_items.append(widgets.VBox([
            widgets.HBox([w_chk, w_info, summary, w_email, w_btn],
                         layout=widgets.Layout(align_items="center")),
        ], layout=widgets.Layout(border="1px solid #ccc", margin="4px 0", padding="6px")))

    s7_container.children = tuple(ui_items)
    s7_status.value = f"<span style='color:green'>✅ {len(ui_items)} reviewers with pending items</span>"


def on_s7_send_all(_):
    s7_btn_send_all.disabled = True
    s7_btn_send_all.description = "Sending..."
    count = 0
    for k, state in s7_row_store.items():
        if state["w_chk"].value and state["w_btn"].description != "Sent":
            # Trigger individual send
            state["w_btn"].click()
            count += 1
            time.sleep(0.5)  # Small delay between sends
    s7_btn_send_all.disabled = False
    s7_btn_send_all.description = f"🔥 Done ({count})"

s7_btn_refresh.on_click(_render_email_rows)
s7_btn_send_all.on_click(on_s7_send_all)

_render_email_rows()


stage7_ui = widgets.VBox([
    widgets.HTML(f"""
        <div style='background: linear-gradient(135deg, #ff9a9e 0%, #fad0c4 100%);
            padding: 20px; border-radius: 8px; color: #333; margin-bottom: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
            <h2 style='margin: 0 0 10px 0;'>📧 Stage 7: Email Notification Center</h2>
            <p style='margin: 0; opacity: 0.85;'>
                Email lookup via AD cache (no per-reviewer API calls).
                Year: {AER_REVIEW_YEAR}. Checkpoint prevents double-send.
            </p>
        </div>
    """),
    s7_subject,
    s7_cc,
    s7_reply_to,
    widgets.HBox([s7_btn_refresh, s7_btn_send_all]),
    s7_status,
    s7_output,
    s7_container,
])

clear_output()
display(stage7_ui)
